<a href="https://colab.research.google.com/github/TiaBerte/rl-for-dfl/blob/main/main_notebook.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

In [2]:
!git clone https://github.com/TiaBerte/rl-for-dfl.git
%cd rl-for-dfl

Cloning into 'rl-for-dfl'...
remote: Enumerating objects: 3644, done.
remote: Counting objects: 100% (3644/3644), done.
remote: Compressing objects: 100% (1114/1114), done.
remote: Total 3644 (delta 1990), reused 3630 (delta 1979), pack-reused 0
Receiving objects: 100% (3644/3644), 4.26 MiB | 14.58 MiB/s, done.
Resolving deltas: 100% (1990/1990), done.
/content/rl-for-dfl


In [3]:
!pip install ortools
!pip install garage
!pip install tabulate

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 79.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 KB 27.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.19.6
    Uninstalling protobuf-3.19.6:
      Successfully uninstalled protobuf-3.19.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.9.2 requires protobuf<3.20,>=3.9.2, but you have protobuf 4.21.12 which is incompatible.
tensorflow-metadata 1.12.0 requires protobuf<4,>=3.13, but you have protobuf 4.21.12 which is incompatible.
tensorboard 2.9.1 requires protobuf<3.20,>=3.9.2, but you have protobuf 4.21.12 which is incompatible.
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/publ

In [4]:
import numpy as np
from usecases.wsmc.generate_instances import generate_training_and_test_sets
from garage.sampler import LocalSampler
from garage.sampler import FragmentWorker
from garage.torch.policies.gaussian_mlp_policy import GaussianMLPPolicy
from garage.torch.policies.tanh_gaussian_mlp_policy import TanhGaussianMLPPolicy
from garage.torch.q_functions.continuous_mlp_q_function import ContinuousMLPQFunction
from garage.experiment import SnapshotConfig
from rl.replay_buffer import ReplayBuffer
from usecases.wsmc.generate_instances import MinSetCoverEnv
from rl.algos import SAC
from garage.torch import set_gpu_mode
from garage.trainer import Trainer
from helpers.garage_utility import CustomEnv, my_wrap_experiment
from garage.envs import GymEnv, normalize
from garage.replay_buffer.path_buffer import PathBuffer
import yaml
import os
import torch
import torch.nn as nn
########################################################################################################################

import warnings
warnings.filterwarnings("ignore")

def train(ctxt: SnapshotConfig = None,
          num_epochs: int = 100,
          batch_size: int = 100,
          env: MinSetCoverEnv = None):
    """
    :param ctxt: garage.experiment.SnapshotConfig: The snapshot configuration used by Trainer to create the
                                                   snapshotter.
    :param num_epochs: int; number of training epochs.
    :param batch_size: int; batch size.
    :param env: usecases.setcover.generate_instances.MinSetCoverEnv; the Minimum Set Cover environment instance.
    :return:
    """

    # Check that the env is not None
    assert env is not None

    # Garage wrapping of a gym environment
    env = CustomEnv(env, max_episode_length=1)
    env.observation_space.dtype = np.float64
    # Replay Buffer used for storing past experience
    replay_buffer = PathBuffer(capacity_in_transitions=int(1e6))

    # A policy represented by a Gaussian distribution which is parameterized by a multilayer perceptron (MLP)
    # policy = GaussianMLPPolicy(env.spec)
    policy = TanhGaussianMLPPolicy(
        env_spec=env.spec,
        hidden_sizes=[256, 256],
        hidden_nonlinearity=nn.ReLU,
        output_nonlinearity=None,
        min_std=np.exp(-20.),
        max_std=np.exp(2.),
    )

    obs, _ = env.reset()
    
    # Q-Network used as critic
    qf1 = ContinuousMLPQFunction(env.spec, hidden_sizes=[256, 256])
    qf2 = ContinuousMLPQFunction(env.spec, hidden_sizes=[256, 256])

    # It's called the "Local" sampler because it runs everything in the same process and thread as where
    # it was called from.
    sampler = LocalSampler(agents=policy,
                            envs=env,
                            max_episode_length=1,
                            worker_class=FragmentWorker)

    # Soft Actor Critic algorithm
    algo = SAC(env_spec=env.spec,
                policy=policy,
                qf1=qf1,
                qf2=qf2,
                replay_buffer=replay_buffer,
                sampler=sampler,
                gradient_steps_per_itr=1
                )

    if torch.cuda.is_available():
        set_gpu_mode(True)
    else:
        set_gpu_mode(False)
        
    algo.to()

    trainer = Trainer(snapshot_config=ctxt)

    trainer.setup(algo=algo, env=env)
    trainer.train(n_epochs=num_epochs, batch_size=batch_size)#, plot=False)

########################################################################################################################


if __name__ == '__main__':
    # Min possible value for the Poisson rates
    MIN_LMBD = 1
    # Max possible value for the Poisson rates
    MAX_LMBD = 10
    # Number of products (elements)
    NUM_PRODS = 5
    # Number of sets (molds)
    NUM_SETS = 25
    # Density of the availability matrix
    DENSITY = 0.02
    # Number of instances to generate
    NUM_INSTANCES = 1000
    # Seed to ensure reproducible results
    SEED = 0
    DATA_PATH = os.path.join('data',
                             'wsmc',
                             f'{NUM_PRODS}x{NUM_SETS}',
                             'linear',
                             f'{NUM_INSTANCES}-instances',
                             f'seed-{SEED}')

    # Set the random seed to ensure reproducibility
    np.random.seed(SEED)
    # Generate training and test set in the specified directory
    generate_training_and_test_sets(data_path=DATA_PATH,
                                    num_instances=NUM_INSTANCES,
                                    num_sets=NUM_SETS,
                                    num_prods=NUM_PRODS,
                                    density=DENSITY,
                                    min_lmbd=MIN_LMBD,
                                    max_lmbd=MAX_LMBD)

    # Set the hyper parameters
    BATCH_SIZE = 100
    EPOCHS = 5000

    SAVEPATH = os.path.join('models',
                            f'{NUM_PRODS}x{NUM_SETS}',
                            f'{NUM_INSTANCES}-instances',
                            f'seed-{SEED}')

    # Create the environment

    env = MinSetCoverEnv(num_prods=NUM_PRODS,
                         num_sets=NUM_SETS,
                         instances_filepath=DATA_PATH,
                         seed=SEED)

    # Create the saving directory if it does not exist
    if not os.path.exists(SAVEPATH):
        os.makedirs(SAVEPATH)

    # Save the configuration params
    config_params = {'batch_size': BATCH_SIZE, 'epochs': EPOCHS}
    with open(os.path.join(SAVEPATH, 'config.yaml'), 'w') as file:
        yaml.dump(config_params, file)

    # Train and test the RL algo
    run = my_wrap_experiment(train,
                             logging_dir=SAVEPATH,
                             snapshot_mode='gap_overwrite',
                             snapshot_gap=EPOCHS // 10,
                             # FIXME: archive_launch_repo=True is not supported
                             archive_launch_repo=False)
    run(num_epochs=EPOCHS, batch_size=BATCH_SIZE, env=env)

/usr/local/lib/python3.8/dist-packages/cma/utilities/utils.py:8: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3, and in 3.10 it will stop working
  from collections import MutableMapping  # since Python 2.4?
/usr/local/lib/python3.8/dist-packages/garage/experiment/deterministic.py:36: UserWarning: Enabeling deterministic mode in PyTorch can have a performance impact when using GPU.
  warnings.warn(


[MinSetCover] - True density: 0.264


Gym environment - Loading instances: 100%|██████████| 1000/1000 [00:00<00:00, 4505.82it/s]


2023-02-06 17:37:32 | [train] Logging to models/5x25/1000-instances/seed-0
2023-02-06 17:37:50 | [train] Obtaining samples...
2023-02-06 17:39:26 | [train] epoch #0 | Saving snapshot...
2023-02-06 17:39:27 | [train] epoch #0 | Saved
2023-02-06 17:39:27 | [train] epoch #0 | Time 96.71 s
2023-02-06 17:39:27 | [train] epoch #0 | EpochTime 96.71 s
----------------------------------  ----------------
AlphaTemperature/mean                    0.9997
Average/TrainAverageReturn          -12839.9
Evaluation/AverageDiscountedReturn   -7570.8
Evaluation/AverageReturn             -7570.8
Evaluation/Iteration                     0
Evaluation/MaxReturn                  -388
Evaluation/MinReturn                -24789
Evaluation/NumEpisodes                  10
Evaluation/StdReturn                  8847.82
Evaluation/TerminationRate               0
Policy/Loss                             -1.75498
QF/Qf1Loss                               2.32219e+08
QF/Qf2Loss                               2.32223e+08
Re

KeyboardInterrupt: ignored